In [9]:
from skan import draw
from skan import Skeleton, summarize
from skan.csr import make_degree_image
import scipy as sc
import skimage as sk
from skimage.io import imsave, imread
from skimage.util import img_as_ubyte
import numpy as np
import pandas as pd
import os
import re
from glob import glob
import joblib
from sklearn.preprocessing import Normalizer
from sklearn.ensemble import RandomForestClassifier
import napari
import itertools
from bioio import BioImage
from pathlib import Path
from random import randint
from matplotlib import pyplot as plt
from scipy import ndimage as ndi
from skimage.morphology import skeletonize, label
from skimage.measure import regionprops
from tqdm import tqdm

In [2]:
def read_img(file):
    return np.squeeze(BioImage(file).get_image_data("ZYX", C=1))

def get_pixel_spacing(file):
    ''' 
    Use one image from dataset read in with BioImage without .get_image_data to access pixel spacing
    '''
    img = BioImage(file)
    pixel_spacing = (img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    #voxel = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    return pixel_spacing

clf = joblib.load('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/models/zebrafish_cell_membranes_V3.joblib')
sigma_min = 1
sigma_max = 8

In [3]:
def get_prediction(file):
    img = read_img(file)
    unstack = np.unstack(img)
    slice_preds = []
    for s in unstack:
        features = sk.feature.multiscale_basic_features(s,sigma_min=sigma_min,sigma_max=sigma_max,channel_axis=None)
        f_flat = features.reshape(-1, features.shape[-1])
        transformer = Normalizer().fit(f_flat)
        norm_f = transformer.transform(f_flat)
        pred = clf.predict(norm_f)
        reshaped_pred = pred.reshape(features.shape[:2])
        slice_preds.append(reshaped_pred)
    pred_stack = np.stack(slice_preds) 
    return img, pred_stack                                   

In [4]:
def get_prediction_list_comp(file):
    img = read_img(file)
    unstack = np.unstack(img)
    features = [sk.feature.multiscale_basic_features(z,sigma_min=sigma_min,sigma_max=sigma_max,channel_axis=None) for z in unstack]
    f_flat = [f.reshape(-1, f.shape[-1]) for f in features]
    transformers = [Normalizer().fit(ff) for ff in f_flat]
    norm_f = [t.transform(ff) for ff, t in zip(f_flat, transformers)]
    preds = [clf.predict(nf) for nf in norm_f]
    reshaped_preds = [p.reshape(f.shape[:2]) for p,f in zip(preds,features)]
    pred_stack = np.stack(reshaped_preds) 
    return img, pred_stack 

In [7]:
def fill_holes(label):
    binary = label - 1
    unstack = np.unstack(binary)
    footprint = sk.morphology.footprint_rectangle((6,6))
    remove_small_objs_pre = [sk.morphology.remove_small_objects(u,max_size=700) for u in unstack]
    dilate = [sk.morphology.dilation(p,footprint,mode="constant",cval=1) for p in remove_small_objs_pre]
    holes = [sk.morphology.remove_small_holes(d) for d in dilate]
    erode = [sk.morphology.erosion(h,footprint,mode="constant",cval=0) for h in holes]
    remove_small_objs = [sk.morphology.remove_small_objects(e,max_size=700) for e in erode]
    gauss = [sk.filters.gaussian(o, sigma=2.0, preserve_range=True) for o in remove_small_objs]
    threshold = [g > 0.25 for g in gauss]
    skeletons = [skeletonize(t) for t in threshold]
    return np.stack(threshold), np.stack(skeletons)

def save_things(path,name,mask,skeletons):
    mask_path = os.path.join(path,'masks')
    os.makedirs(mask_path,exist_ok=True)
    skeleton_path = os.path.join(path,'skeletons')
    os.makedirs(skeleton_path,exist_ok=True)
    mask_bit = np.array(mask,dtype='uint8')
    skeletons_bit = np.array(skeletons,dtype='uint8')
    imsave(os.path.join(mask_path,f'{name[:-4]}_mask.tif'),mask_bit,check_contrast=False)
    imsave(os.path.join(skeleton_path,f'{name[:-4]}_skeleton.tif'),skeletons_bit,check_contrast=False)

In [6]:
files = sorted(glob('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Images/*/*.czi'))
#imgs = list(map(read_img,files))
pixel_spacing = list(map(get_pixel_spacing,files))

In [ ]:
for f in tqdm(files):
    name = os.path.basename(f)
    path = Path(f).parent
    img, preds = get_prediction_list_comp(f)
    mask, skeletons = fill_holes(preds)
    save_things(path,name,mask,skeletons)

In [ ]:
skeleton_files = sorted(glob('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Images/*/skeletons/*.tif'))

degree_imgs = []
for sf in skeleton_files:
    path = Path(sf).parents[1]
    name = os.path.basename(sf)[:-4]
    new_name = name.replace('skeleton','deg_img')
    os.makedirs(os.path.join(path,'degree_imgs'),exist_ok=True)
    skimg = imread(sf)
    sk_unstack = np.unstack(skimg)
    deg_imgs = list(map(make_degree_image,sk_unstack))
    degree_imgs.append(deg_imgs)
    deg_stack = np.stack(deg_imgs,dtype='int8')
    imsave(os.path.join(path,'degree_imgs',f'{new_name}.tif'),deg_stack,check_contrast=False)


In [14]:
v = napari.Viewer()

In [16]:
deg_stack = np.stack(degree_imgs[0])
v.add_labels(deg_stack == 2)

<Labels layer 'Labels [1]' at 0x7cb981746960>

## Testing Cells

In [ ]:
# Getting measurements for skeleton components


In [ ]:
imgs = []
preds = []
i = randint(0,len(files))
#img, pred = get_prediction(files[i])

#for f in files:
#    img, pred = get_prediction(f)
#    imgs.append(img)
#    preds.append(pred)
#    break

In [ ]:
img, pred = get_prediction_list_comp(files[i])

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(imgs[0])
v.add_labels(preds[0]==2)

In [ ]:
pred = preds[0]
img = imgs[0]

In [ ]:
binary = pred - 1
binary_unstack = np.unstack(binary)

In [ ]:
v.add_labels(binary)

In [ ]:
footprint = sk.morphology.footprint_rectangle((6,6))
remove_small_objs_pre = [sk.morphology.remove_small_objects(z,max_size=700) for z in binary_unstack]
dilate = [sk.morphology.dilation(p,footprint,mode="constant",cval=1) for p in remove_small_objs_pre]
holes = [sk.morphology.remove_small_holes(d) for d in dilate]
erode = [sk.morphology.erosion(h,footprint,mode="constant",cval=0) for h in holes]
remove_small_objs = [sk.morphology.remove_small_objects(e,max_size=700) for e in erode]


In [ ]:
dilate_stack = np.stack(dilate)
holes_stack = np.stack(holes)
erode_stack = np.stack(erode)
objs_stack = np.stack(remove_small_objs)

In [ ]:
#v.add_labels(dilate_stack)
#v.add_labels(holes_stack)
#v.add_labels(erode_stack)
v.add_image(img)
v.add_labels(objs_stack)

In [ ]:
skeleton = [skeletonize(z) for z in remove_small_objs]
skeleton_stack = np.stack(skeleton)

In [ ]:
v.add_labels(skeleton_stack)

In [ ]:
v.add_image(img)

In [ ]:
add = [z * 255 for z in remove_small_objs]

In [ ]:
gauss = [sk.filters.gaussian(o, sigma=2.0, preserve_range=True) for o in remove_small_objs]
gauss_stack = np.stack(gauss)

In [ ]:
v.add_image(gauss_stack)

In [ ]:
threshold = gauss_stack > 0.25

In [ ]:
unstack_threshold = np.unstack(threshold)

In [ ]:
skeleton_gauss = [skeletonize(t) for t in unstack_threshold]
skeleton_gauss_stack = np.stack(skeleton_gauss)

In [ ]:
#v.add_labels(threshold)
v.add_labels(skeleton_gauss_stack)
#v.add_labels(skeleton_stack)

In [ ]:
len(skeleton_gauss)

In [ ]:
scale = pixel_spacing[0][0]
scale

In [ ]:
skeleton_summary = [summarize(Skeleton(s,spacing=scale),separator='_') for s in skeleton_gauss]

In [ ]:
skeleton_summary[0].head()

In [ ]:
degree_images = list(map(make_degree_image,skeleton_gauss))

In [ ]:
degree_images_stack = np.stack(degree_images)

In [ ]:
v.add_labels(degree_images_stack)

In [ ]:
v.add_image(img)